# DistilBERT: A Distilled Version of BERT

DistilBERT is a compact and efficient version of BERT (Bidirectional Encoder Representations from Transformers) that maintains much of the original model's performance while significantly reducing its size and computational requirements [1].

## Key Mathematical Aspects

1. **Knowledge Distillation**
   DistilBERT uses knowledge distillation, a compression technique where a smaller model (student) is trained to mimic a larger model (teacher) [2]. The objective function for distillation is:

   $L_{distill} = \sum_{i} t_i * \log(s_i)$

   where $t_i$ and $s_i$ are the teacher's and student's predicted probabilities for class $i$, respectively.

2. **Triple Loss Function**
   DistilBERT's training incorporates a triple loss function:

   $L = \alpha * L_{ce} + \beta * L_{distill} + \gamma * L_{cosine}$

   - $L_{ce}$: masked language modeling loss (cross-entropy)
   - $L_{distill}$: distillation loss
   - $L_{cosine}$: cosine embedding loss to align hidden states
   - $\alpha$, $\beta$, $\gamma$: weighting coefficients

3. **Architectural Changes**
   - Reduced layers: 6 layers (vs. 12 in BERT-base)
   - Removed token-type embeddings and pooler
   - Kept other dimensions same as BERT-base (hidden size 768, 12 attention heads)

4. **Initialization and Training**
   - Initialize from every other layer of the teacher model
   - Use dynamic masking instead of static masking
   - Train on large batches using gradient accumulation

5. **Efficiency Gains**
   - 40% smaller
   - 60% faster
   - Retains 97% of BERT's performance on GLUE benchmark

6. **Attention Mechanism**
   DistilBERT retains the multi-head attention mechanism:

   $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$

   where $Q$, $K$, and $V$ are query, key, and value matrices, and $d_k$ is the dimension of the key vectors.

7. **Positional Encodings**
   Uses sinusoidal positional encodings:

   $PE_{(pos,2i)} = sin(pos / 10000^{2i/d_{model}})$
   $PE_{(pos,2i+1)} = cos(pos / 10000^{2i/d_{model}})$

   where $pos$ is the position and $i$ is the dimension.

References:
[1] Sanh, V., Debut, L., Chaumond, J., & Wolf, T. (2019). DistilBERT, a distilled version of BERT: smaller, faster, cheaper and lighter. arXiv preprint arXiv:1910.01108.
[2] Hinton, G., Vinyals, O., & Dean, J. (2015). Distilling the knowledge in a neural network. arXiv preprint arXiv:1503.02531.
[3] Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., ... & Polosukhin, I. (2017). Attention is all you need. In Advances in neural information processing systems (pp. 5998-6008).

In [4]:
!pip install torch==2.3.1+cu121 transformers==4.38.2 datasets==2.17.1

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
import math
from transformers import BertTokenizer, BertForSequenceClassification

# Simplified DistilBERT model
class DistilBERTLayer(nn.Module):
    def __init__(self, d_model, nhead, dim_feedforward):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, nhead)
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(0.1)

    def forward(self, src):
        src2 = self.self_attn(src, src, src)[0]
        src = src + self.dropout(src2)
        src = self.norm1(src)
        src2 = self.linear2(F.relu(self.linear1(src)))
        src = src + self.dropout(src2)
        src = self.norm2(src)
        return src

class DistilBERT(nn.Module):
    def __init__(self, vocab_size, d_model, nhead, dim_feedforward, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.position_encoding = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([DistilBERTLayer(d_model, nhead, dim_feedforward) for _ in range(num_layers)])
        self.classifier = nn.Linear(d_model, 2)  # Binary classification

    def forward(self, x):
        x = self.embedding(x)
        x = self.position_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.classifier(x.mean(dim=1))

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:x.size(0), :]

# Simple tokenizer
class SimpleTokenizer:
    def __init__(self):
        self.word_to_idx = {"[PAD]": 0, "[UNK]": 1}
        self.idx_to_word = {0: "[PAD]", 1: "[UNK]"}
        self.vocab_size = 2

    def fit(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word_to_idx:
                    self.word_to_idx[word] = self.vocab_size
                    self.idx_to_word[self.vocab_size] = word
                    self.vocab_size += 1

    def encode(self, text, max_length):
        tokens = [self.word_to_idx.get(word, 1) for word in text.split()]
        if len(tokens) < max_length:
            tokens += [0] * (max_length - len(tokens))
        else:
            tokens = tokens[:max_length]
        return tokens

# Dataset
class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoded = self.tokenizer.encode(text, self.max_length)
        return torch.tensor(encoded), torch.tensor(label)

# Load data and prepare dataset
dataset = load_dataset("imdb", split="train[:1000]")
texts = dataset["text"]
labels = dataset["label"]

tokenizer = SimpleTokenizer()
tokenizer.fit(texts)

max_length = 128
dataset = IMDBDataset(texts, labels, tokenizer, max_length)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# Initialize student model (our simplified DistilBERT)
vocab_size = tokenizer.vocab_size
d_model = 256
nhead = 4
dim_feedforward = 1024
num_layers = 3

student_model = DistilBERT(vocab_size, d_model, nhead, dim_feedforward, num_layers)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student_model.to(device)

# Load pre-trained teacher model (BERT)
teacher_model = BertForSequenceClassification.from_pretrained('bert-base-uncased')
teacher_model.to(device)
teacher_model.eval()

# BERT tokenizer for teacher model
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Loss functions
criterion = nn.CrossEntropyLoss()
distillation_loss = nn.KLDivLoss(reduction='batchmean')

# Optimizer
optimizer = torch.optim.Adam(student_model.parameters(), lr=1e-4)

# Training loop with distillation
num_epochs = 3
alpha = 0.5  # Weight for distillation loss

for epoch in range(num_epochs):
    student_model.train()
    for batch in dataloader:
        inputs, labels = batch
        inputs, labels = inputs.to(device), labels.to(device)

        # Forward pass through student model
        student_outputs = student_model(inputs)

        # Forward pass through teacher model
        with torch.no_grad():
            bert_inputs = bert_tokenizer([" ".join([tokenizer.idx_to_word[idx.item()] for idx in input if idx.item() != 0]) for input in inputs], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
            teacher_outputs = teacher_model(**bert_inputs).logits

        # Calculate losses
        student_loss = criterion(student_outputs, labels)
        distill_loss = distillation_loss(F.log_softmax(student_outputs / 2.0, dim=1),
                                         F.softmax(teacher_outputs / 2.0, dim=1))

        # Combine losses
        loss = alpha * student_loss + (1 - alpha) * distill_loss

        # Backpropagation and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f"Epoch {epoch+1}/{num_epochs} completed")

print("Training completed!")

# Inference example
test_text = "This movie was fantastic! I really enjoyed it."
encoded_test = torch.tensor([tokenizer.encode(test_text, max_length)]).to(device)
student_model.eval()
with torch.no_grad():
    output = student_model(encoded_test)
    predicted_class = torch.argmax(output, dim=1).item()

print(f"Test text: {test_text}")
print(f"Predicted sentiment: {'Positive' if predicted_class == 1 else 'Negative'}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3 completed
Epoch 2/3 completed
Epoch 3/3 completed
Training completed!
Test text: This movie was fantastic! I really enjoyed it.
Predicted sentiment: Negative


# Mathematical Aspects of Knowledge Distillation

Knowledge distillation is a technique used to transfer knowledge from a large model (teacher) to a smaller model (student). Here are the key mathematical aspects of this process:

1. **Temperature Scaling**: We use a temperature parameter T to "soften" the probability distributions:
   $softmax(z_i, T) = \frac{\exp(z_i/T)}{\sum_j \exp(z_j/T)}$

   In our implementation:
```python
   student_soft = F.log_softmax(student_outputs / 2.0, dim=1)
   teacher_soft = F.softmax(teacher_outputs / 2.0, dim=1)

2. **Kullback-Leibler Divergence**: The distillation loss is calculated using the KL divergence:

   $KL(P||Q) = \sum_i P(i) * \log(\frac{P(i)}{Q(i)})$

   Implemented as:
```python
   distillation_loss = nn.KLDivLoss(reduction='batchmean')
   distill_loss = distillation_loss(student_soft, teacher_soft)
   ```

3. **Loss Combination**: The final loss is a weighted combination of cross-entropy and distillation loss:

   $L = \alpha * L_{CE} + (1 - \alpha) * L_{KL}$

  ```python
    loss = alpha * student_loss + (1 - alpha) * distill_loss
  ```
4. **Gradient Flow**: During backpropagation, gradients flow through both loss components:

   $\nabla_\theta L = \alpha * \nabla_\theta L_{CE} + (1 - \alpha) * \nabla_\theta L_{KL}$

5. **Optimization**: We use Adam optimizer to update the student model's parameters.

6. **Information Transfer**: Soft targets from the teacher provide more nuanced information than hard labels.

7. **Dark Knowledge**: The ratios of small probabilities in the teacher's soft predictions can be highly informative.

8. **Temperature Annealing**: Although not implemented here, some techniques gradually decrease T during training:
```python
   def anneal_temperature(epoch, total_epochs, initial_T, final_T):
       return initial_T - (initial_T - final_T) * (epoch / total_epochs)

In summary, this distillation process allows the student model to learn a more nuanced decision boundary by leveraging the teacher's knowledge, potentially leading to better performance than training on hard labels alone.